# Kaggle – Data on the top
Tu profe ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Métrica: RMSE

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

Donde $y_i$ es el valor real y $\hat{y}_i$ es el valor predicho. **Cuanto menor, mejor.**

---
# PARTE 1: Entrenamiento del modelo

## 1. Librerías

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

## 2. Datos

In [2]:
df = pd.read_csv('./data/train.csv', encoding='latin-1'
                 )

### 2.1 Exploración de los datos

In [3]:
# Tu código aquí
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         912 non-null    int64  
 1   Company           912 non-null    object 
 2   Product           912 non-null    object 
 3   TypeName          912 non-null    object 
 4   Inches            912 non-null    float64
 5   ScreenResolution  912 non-null    object 
 6   Cpu               912 non-null    object 
 7   Ram               912 non-null    object 
 8   Memory            912 non-null    object 
 9   Gpu               912 non-null    object 
 10  OpSys             912 non-null    object 
 11  Weight            912 non-null    object 
 12  Price_in_euros    912 non-null    float64
dtypes: float64(2), int64(1), object(10)
memory usage: 92.8+ KB


In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
laptop_ID,912.0,650.312500,382.727748,2.0,324.75,636.5,982.2500,1320.0
Inches,912.0,14.981579,1.436719,10.1,14.00,15.6,15.6000,18.4
Price_in_euros,912.0,1111.724090,687.959172,174.0,589.00,978.0,1483.9425,6099.0


In [5]:
df.head(5)

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
0,755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
1,618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
2,909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
3,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
4,286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


In [6]:
for col in df.columns:
    print(df[col].value_counts())

laptop_ID
755     1
618     1
909     1
2       1
286     1
       ..
28      1
1160    1
78      1
23      1
229     1
Name: count, Length: 912, dtype: int64
Company
Lenovo       202
Dell         197
HP           194
Asus         121
Acer          74
MSI           37
Toshiba       34
Apple         17
Razer          6
Mediacom       6
Samsung        5
Microsoft      5
Xiaomi         3
Huawei         2
Chuwi          2
Google         2
Vero           2
Fujitsu        2
LG             1
Name: count, dtype: int64
Product
XPS 13                                   23
Inspiron 3567                            22
Legion Y520-15IKBN                       15
Vostro 3568                              14
ProBook 450                              13
                                         ..
Inspiron 7773                             1
ENVY -                                    1
Latitude E7270                            1
Rog GL552VW-CN470T                        1
15-BS026nv (i5-7200U/8GB/256GB/Radeo

### 2.2 Definir X e y


In [ ]:
# Tu código aquí
#Renombramos target
df.rename(columns={'Price_in_euros': 'target'}, inplace=True)


### 2.3 Dividir en train y test

In [8]:
# Tu código aquí
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=['target']),
    df['target'],
    test_size=0.2,
    random_state=42
)

## 3. Procesado de datos

> 🚨 **Data leakage:** si usas un scaler, haz **`.fit()` SOLO sobre `X_train`** y luego aplica `.transform()` sobre `X_train` y `X_test` por separado.
>
> Recuerda también que **todo lo que hagas aquí deberás replicarlo después en `test.csv`** (sección 6).

In [9]:
X_train['Ram_final'] = X_train["Ram"].str.replace("GB", "").astype(int)
X_test['Ram_final'] = X_test["Ram"].str.replace("GB", "").astype(int)

# Weight (igual)
X_train['Weight_kg'] = X_train["Weight"].str.replace("kg", "").astype(float)
X_test['Weight_kg'] = X_test["Weight"].str.replace("kg", "").astype(float)

# Dropar las originales
X_train = X_train.drop(columns=['Ram', 'Weight'])
X_test = X_test.drop(columns=['Ram', 'Weight'])

In [10]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

cat_cols = ['Company', 'Product', 'TypeName', 'OpSys', 'ScreenResolution', 'Cpu', 'Memory', 'Gpu']

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

encoder.fit(X_train[cat_cols])

encoded_train = encoder.transform(X_train[cat_cols])
encoded_test = encoder.transform(X_test[cat_cols])

encoded_cols = encoder.get_feature_names_out(cat_cols)

X_train_enc = pd.DataFrame(encoded_train, columns=encoded_cols, index=X_train.index)
X_test_enc = pd.DataFrame(encoded_test, columns=encoded_cols, index=X_test.index)

X_train = X_train.drop(columns=cat_cols).join(X_train_enc)
X_test = X_test.drop(columns=cat_cols).join(X_test_enc)

In [11]:
X_train = X_train.drop(columns=['laptop_ID'])
X_test = X_test.drop(columns=['laptop_ID'])

In [14]:
X_train

,Inches,Ram_final,Weight_kg,Company_Acer,Company_Apple,Company_Asus,Company_Chuwi,Company_Dell,Company_Fujitsu,Company_Google,...,Gpu_Nvidia GeForce GTX 980,Gpu_Nvidia GeForce GTX 980M,Gpu_Nvidia GeForce MX130,Gpu_Nvidia GeForce MX150,Gpu_Nvidia Quadro M1000M,Gpu_Nvidia Quadro M1200,Gpu_Nvidia Quadro M2000M,Gpu_Nvidia Quadro M2200M,Gpu_Nvidia Quadro M500M,Gpu_Nvidia Quadro M620
25,17.3,8,3.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
84,15.6,16,2.56,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10,13.3,8,1.37,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
342,14.0,4,1.54,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
890,17.3,16,2.80,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,14.0,8,1.94,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
270,15.6,6,2.20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
860,12.5,16,1.18,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
435,15.6,4,2.20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
from sklearn.preprocessing import StandardScaler

num_cols = ['Inches', 'Ram_final', 'Weight_kg']

scaler = StandardScaler()

# Fit solo en train
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

## 4. Modelado

### 4.1 Entrenamiento

In [16]:
# Tu código aquí
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

model_lr = LinearRegression()
model_lr.fit(X_train, y_train)

y_pred_lr = model_lr.predict(X_test)

rmse_lr = root_mean_squared_error(y_test, y_pred_lr)
print(f"RMSE Linear Regression: {rmse_lr:.2f}")

RMSE Linear Regression: 379.77


In [17]:
from sklearn.linear_model import Ridge

model_ridge = Ridge(alpha=1.0)
model_ridge.fit(X_train, y_train)

y_pred_ridge = model_ridge.predict(X_test)

rmse_ridge = root_mean_squared_error(y_test, y_pred_ridge)
print(f"RMSE Ridge: {rmse_ridge:.2f}")

RMSE Ridge: 309.56


In [18]:
from sklearn.ensemble import RandomForestRegressor

model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)

rmse_rf = root_mean_squared_error(y_test, y_pred_rf)
print(f"RMSE Random Forest: {rmse_rf:.2f}")

RMSE Random Forest: 372.07


In [19]:
X_train.columns = X_train.columns.str.replace(r'[\[\]<]', '', regex=True)
X_test.columns = X_test.columns.str.replace(r'[\[\]<]', '', regex=True)

In [20]:
from xgboost import XGBRegressor

model_xgb = XGBRegressor(n_estimators=100, random_state=42)
model_xgb.fit(X_train, y_train)

y_pred_xgb = model_xgb.predict(X_test)

rmse_xgb = root_mean_squared_error(y_test, y_pred_xgb)
print(f"RMSE XGBoost: {rmse_xgb:.2f}")

RMSE XGBoost: 306.85


In [26]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
}

grid_search = GridSearchCV(
    XGBRegressor(random_state=42),
    param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_search.best_params_}")

y_pred_grid = grid_search.best_estimator_.predict(X_test)
rmse_grid = root_mean_squared_error(y_test, y_pred_grid)
print(f"RMSE XGBoost tuneado: {rmse_grid:.2f}")

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Mejores parámetros: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}


AttributeError: 'DataFrame' object has no attribute 'dtype'

In [23]:
import re

X_train.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in X_train.columns]
X_test.columns = [re.sub(r'[^A-Za-z0-9_]', '', col) for col in X_test.columns]

In [24]:
from lightgbm import LGBMRegressor

model_lgbm = LGBMRegressor(n_estimators=100, random_state=42)
model_lgbm.fit(X_train, y_train)

y_pred_lgbm = model_lgbm.predict(X_test)

rmse_lgbm = root_mean_squared_error(y_test, y_pred_lgbm)
print(f"RMSE LightGBM: {rmse_lgbm:.2f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000987 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 206
[LightGBM] [Info] Number of data points in the train set: 729, number of used features: 52
[LightGBM] [Info] Start training from score 1103.789314
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

### 4.2 Métricas

Recuerda que en la competición se evalúa con **RMSE**.

In [ ]:
# Tu código aquí


### 4.3 Optimización (up to you 🫰🏻)

In [ ]:
# Tu código aquí


## 5. Reentrenamiento sobre todos los datos de `train.csv`

Una vez afinado el modelo, reentrenamos con **todos** los datos disponibles antes de predecir sobre `test.csv`.

> ¿Por qué? El split anterior era solo para validar localmente. Para la submission final queremos aprovechar el 100% de los datos de entrenamiento.

In [27]:
# Tu código aquí
# Usamos el df original completo (sin split)
df_full = pd.read_csv('./data/train.csv', encoding='latin-1')
df_full.rename(columns={'Price_in_euros': 'target'}, inplace=True)

X_full = df_full.drop(columns=['target', 'laptop_ID'])
y_full = df_full['target']

# Ram y Weight
X_full['Ram_final'] = X_full["Ram"].str.replace("GB", "").astype(int)
X_full['Weight_kg'] = X_full["Weight"].str.replace("kg", "").astype(float)
X_full = X_full.drop(columns=['Ram', 'Weight'])

# One Hot Encoder - fit sobre todos los datos de train
cat_cols = ['Company', 'Product', 'TypeName', 'OpSys', 'ScreenResolution', 'Cpu', 'Memory', 'Gpu']
encoder_final = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder_final.fit(X_full[cat_cols])
encoded = encoder_final.transform(X_full[cat_cols])
encoded_cols = encoder_final.get_feature_names_out(cat_cols)
X_full = X_full.drop(columns=cat_cols).join(pd.DataFrame(encoded, columns=encoded_cols, index=X_full.index))

# Limpiar nombres de columnas
X_full.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in X_full.columns]

# Scaler - fit sobre todos los datos de train
num_cols = ['Inches', 'Ram_final', 'Weight_kg']
scaler_final = StandardScaler()
X_full[num_cols] = scaler_final.fit_transform(X_full[num_cols])

# Reentrenar con los mejores parámetros del GridSearch
model_final = XGBRegressor(**grid_search.best_params_, random_state=42)
model_final.fit(X_full, y_full)
print("Modelo reentrenado con todos los datos de train ✅")

Modelo reentrenado con todos los datos de train ✅


---
# PARTE 2: Predicción y submission

Una vez tengas el modelo listo, toca predecir sobre `test.csv` y generar el archivo de submission.

## 6. Carga los datos de `test.csv`

In [28]:
X_pred = pd.read_csv('./data/test.csv', encoding='latin-1')
X_pred.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1,1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
2,1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
3,1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
4,1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg


## 7. Replica el procesado en `test.csv`

> ⚠️ Usa `.transform()`, **nunca `.fit_transform()`** sobre los datos de test.
>
> Lo único que **no puedes hacer** es eliminar filas.

In [29]:
# Tu código aquí
# Ram y Weight
X_pred['Ram_final'] = X_pred["Ram"].str.replace("GB", "").astype(int)
X_pred['Weight_kg'] = X_pred["Weight"].str.replace("kg", "").astype(float)
X_pred = X_pred.drop(columns=['Ram', 'Weight', 'laptop_ID'])

# One Hot - solo transform
encoded_pred = encoder_final.transform(X_pred[cat_cols])
X_pred_enc = pd.DataFrame(encoded_pred, columns=encoded_cols, index=X_pred.index)
X_pred = X_pred.drop(columns=cat_cols).join(X_pred_enc)

# Limpiar nombres
X_pred.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in X_pred.columns]

# Scaler - solo transform
X_pred[num_cols] = scaler_final.transform(X_pred[num_cols])

## 8. Genera la submission

### 8.1 ¿Qué formato espera Kaggle?

In [30]:
sample = pd.read_csv('./data/sample_submission.csv', encoding='latin-1')
sample.head()

,laptop_ID,Price_in_euros
0,209,1949.1
1,1281,805.0
2,1168,1101.0
3,1231,1293.8
4,1020,1832.6


### 8.2 Crea tu submission

In [31]:
# Tu código aquí
# 8.2 Crea tu submission
y_pred_final = model_final.predict(X_pred)

submission = pd.DataFrame({
    'laptop_ID': pd.read_csv('./data/test.csv', encoding='latin-1')['laptop_ID'],
    'Price_in_euros': y_pred_final
})

submission.head()

,laptop_ID,Price_in_euros
0,209,1608.701172
1,1281,293.258759
2,1168,361.846344
3,1231,1150.492554
4,1020,935.683594


### 8.3 Chequeador

Pásale el chequeador antes de subir a Kaggle. Si todo está bien, guardará el CSV automáticamente con un nombre único.

In [32]:
def checker(df_to_submit, sample, filename=None):
    """
    Valida que tu submission tenga la forma requerida por Kaggle.
    Si es correcta, guarda el CSV listo para subir.
    Si no, lee el mensaje de error y corrígelo.
    """
    if df_to_submit.shape != sample.shape:
        print(' Shape incorrecto.')
        print(f'   Tu submission: {df_to_submit.shape} | Esperado: {sample.shape}')
        print('   Revisa que no hayas borrado filas del test ni añadido/quitado columnas.')
        return

    if not (df_to_submit.columns == sample.columns).all():
        print(' Nombres de columnas incorrectos.')
        print(f'   Tus columnas:       {list(df_to_submit.columns)}')
        print(f'   Columnas esperadas: {list(sample.columns)}')
        return

    if not (df_to_submit['laptop_ID'] == sample['laptop_ID']).all():
        print(' Los IDs no coinciden con sample_submission. Revisa que no hayas reordenado el test.csv.')
        return

    if filename is None:
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'submission_{timestamp}.csv'

    df_to_submit.to_csv(filename, index=False)
    print(f" ¡Todo correcto! Submission guardada como '{filename}'. ¡A Kaggle!")

In [34]:
sample = pd.read_csv('./data/sample_submission.csv', encoding='latin-1')
checker(submission, sample)

 ¡Todo correcto! Submission guardada como 'submission_20260629_172740.csv'. ¡A Kaggle!
